In [ ]:
# Check and install required dependencies (PyTorch & tqdm)
try:
    # Try importing required libraries
    import torch
    import torchvision
    import tqdm

    # If successful, libraries are already available
    print("PyTorch, torchvision, and tqdm are already installed.")

except Exception:
    # If import fails, install PyTorch with CUDA 12.1 support
    # (suitable for most modern GPUs on Google Colab)
    %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

    # Install tqdm for progress visualization
    %pip install -q tqdm


PyTorch/torchvision already installed.


In [ ]:
#@title 🔧 Imports, seeds, device
import os
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms
from tqdm import tqdm

# Reproducibility: Set random seeds

def seed_everything(seed: int = 42):
    """
    Set random seeds for Python, NumPy, and PyTorch to ensure
    reproducibility of experimental results.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
seed_everything(123)


# Device configuration
# Use GPU if available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [ ]:
# Install gdown for downloading files from Google Drive
%pip install gdown

In [ ]:
#@title ⚙️ Configuration

import os


class CFG:
    # -----------------------------------------------------
    # Dataset 
    # -----------------------------------------------------
    source_name = "NWPU"
    target_name = "NaSC-TG2-RGB"
    num_classes = 10  # Number of shared classes between source and target domains
    # -----------------------------------------------------
    # Source training 
    # -----------------------------------------------------
    src_epochs = 15           # Try 10–20 for stronger results
    src_batch_size = 64
    src_lr = 1e-2

    # -----------------------------------------------------
    # Source-free adaptation
    # -----------------------------------------------------
    sf_epochs = 15             # Try 15–30 for stronger results
    sf_batch_size = 64
    sf_lr = 1e-3                # Lower LR for backbone fine-tuning


    # -----------------------------------------------------
    # Pseudo-labeling & confidence control
    # -----------------------------------------------------
    conf_thresh = 0.85          # Relaxed threshold to include more samples
                                 # (important for confusing classes like Residential)


    # -----------------------------------------------------
    # Loss weighting (CAB-SFDA)
    # -----------------------------------------------------
    lambda_im = 1.0             # Information maximization loss
    lambda_cons = 1.0           # Weak-strong consistency loss
    lambda_pl = 1.0             # Pseudo-label self-training loss
    lambda_proto = 0.5          # Prototype attraction loss


    # -----------------------------------------------------
    # Consistency regularization
    # -----------------------------------------------------
    temp_cons = 0.5             # Temperature for consistency sharpening
    rampup_epochs = 50          # Gradual ramp-up for consistency & pseudo-label losses


    # -----------------------------------------------------
    # Prototype memory settings
    # -----------------------------------------------------
    proto_momentum = 0.9        # Slow updates for stable prototype memory
    feat_dim = 128              # Feature dimension of bottleneck layer


    # -----------------------------------------------------
    # Miscellaneous
    # -----------------------------------------------------
    num_workers = 4
    out_dir = "./cab_sfda_ckpts_nwpu_to_NaSC-TG2-RGB"


# ---------------------------------------------------------
# Create output directory
# ---------------------------------------------------------
os.makedirs(CFG.out_dir, exist_ok=True)

print("✅ Configuration loaded: CAB-SFDA (NWPU → NaSC-TG2-RGB)")


✅ Config updated for stronger CAB-SFDA performance


In [ ]:
# =========================================================
# 📂 Dataset Preparation & Train/Test Split
# (NWPU-RESISC45 → NaSC-TG2-RGB)
# =========================================================
import os
import shutil
import random
import glob 
import stat # Added for file permissions handling


def prepare_and_split_data(src_root, dst_root, class_map, split_ratio=0.7, copy_method='copy'):
    """
    Prepares a dataset by mapping classes and splitting into train/test sets.

    Args:
        src_root: Path to the original dataset with class folders.
        dst_root: Path for the output split dataset (will contain /train and /test).
        class_map: Dict mapping canonical class names to actual folder names in src_root.
        split_ratio: Fraction of images to use for the 'train' phase.
        copy_method: 'copy' for source domain or 'symlink' for target domain.
    """
    print(f"\n--- Preparing Dataset: {os.path.basename(src_root)} ---")
    random.seed(42) 

    for canonical_cls, folder_name in class_map.items():
        cls_path = os.path.join(src_root, folder_name)

        if not os.path.isdir(cls_path):
            print(f"⚠️ Missing expected class folder: {cls_path}")
            continue

        # Use glob for file matching (looks for common image extensions recursively)
        img_pattern = os.path.join(cls_path, '*')
        imgs = [f for f in glob.glob(img_pattern, recursive=True) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif'))]
        
        if not imgs:
            print(f"⚠️ No images found in: {cls_path}")
            continue

        random.shuffle(imgs)
        split_idx = int(len(imgs) * split_ratio)
        train_imgs = imgs[:split_idx]
        test_imgs = imgs[split_idx:]
        # Create train/test folders and copy data
        for phase, img_list in zip(['train', 'test'], [train_imgs, test_imgs]):
            out_dir = os.path.join(dst_root, phase, canonical_cls)
            os.makedirs(out_dir, exist_ok=True)
            
            for img_src in img_list:
                img_dst = os.path.join(out_dir, os.path.basename(img_src))
                
                try:
                    if copy_method == 'symlink':
                        if not os.path.exists(img_dst):
                            os.symlink(img_src, img_dst)
                    else:
                        # Ensures write permission on the source file before copying
                        os.chmod(img_src, stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
                        shutil.copy2(img_src, img_dst)
                
                except PermissionError:
                    print(f"🚨 PERMISSION DENIED: Failed to copy/link file.")
                    print(f"   Source: {img_src}")
                    print(f"   Destination: {img_dst}")
                    print(f"   TRY: Running your notebook/terminal as Administrator.")
                    continue # Skip this file and try the next one
                except Exception as e:
                    print(f"🚨 UNEXPECTED ERROR during file operation: {e}")
                    continue
        
        print(f"✅ Class '{canonical_cls}' (Total: {len(imgs)}) -> Train: {len(train_imgs)}, Test: {len(test_imgs)}")

    print(f"\n✅ Dataset preparation completed. Saved to: {dst_root}")


# --- 2. CLASS MAPPINGS (NWPU-RESISC45 -> NaSC-TG2) ---

# Canonical classes (10 common classes for this pair)
canonical_classes = [
    'Beach', 'CircularFarmland', 'Cloud', 'Desert', 'Forest',
    'Mountain', 'Rectangular farmland', 'Residential', 'River', 'Snowberg'
]

# Source: NWPU-RESISC45 folder names
nwpu_folder_map = {
    'Beach': 'beach',
    'CircularFarmland': 'circular_farmland',
    'Cloud': 'cloud',
    'Desert': 'desert',
    'Forest': 'forest',
    'Mountain': 'mountain',
    'Rectangular farmland': 'rectangular_farmland',
    'Residential': 'dense_residential', # NWPU uses 'dense_residential'
    'River': 'river',
    'Snowberg': 'snowberg'
}

# Target: NaSC-TG2 folder names
nasc_folder_map = {
    'Beach': 'beach',
    'CircularFarmland': 'circularfarmland',
    'Cloud': 'cloud',
    'Desert': 'desert',
    'Forest': 'forest',
    'Mountain': 'mountain',
    'Rectangular farmland': 'rectangularfarmland',
    'Residential': 'residential', # NaSC uses 'residential'
    'River': 'river',
    'Snowberg': 'snowberg'
}


# --- 3. EXECUTION ---

# A) SOURCE DOMAIN (NWPU-RESISC45)
print("\n[STEP A: Preparing Source Domain (NWPU-RESISC45)]")
prepare_and_split_data(
    src_root="./data/NWPU-RESISC45/NWPU-RESISC45", # <--- CHECK/ADJUST THIS PATH
    dst_root="./data/source_nwpu_split", 
    class_map=nwpu_folder_map, 
    split_ratio=0.7, 
    copy_method='copy'
)

# B) TARGET DOMAIN (NaSC-TG2)
print("\n[STEP B: Preparing Target Domain (NaSC-TG2)]")
prepare_and_split_data(
    src_root="./data/NaSC-TG2-RGB", # <--- CHECK/ADJUST THIS PATH
    dst_root="./data/target_nasc_split", 
    class_map=nasc_folder_map, 
    split_ratio=0.7, 
    copy_method='copy' # Recommended for efficiency, but use 'copy' if symlink fails
)

In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
import os
from PIL import Image

# 🌍 Normalization for remote sensing RGB images
# Using standard ImageNet mean/std is fine for pre-trained models
mean_rgb = (0.485, 0.456, 0.406)
std_rgb  = (0.229, 0.224, 0.225)

# --- TARGET (NaSC-TG2) TRANSFORMS ---

# 🟢 Weak augmentation for target (minimal distortion)
# Only resize and normalize. This view generates the pseudo-labels.
tgt_weak_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean_rgb, std_rgb),
])

# 🔴 Strong augmentation for target (stabilized for consistency)
# Milder spatial and color changes to preserve scene semantics.
tgt_strong_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    # Milder Crop: Reduced scale range for RandomResizedCrop
    transforms.RandomResizedCrop(256, scale=(0.9, 1.0), ratio=(0.9, 1.1)), 
    
    transforms.RandomHorizontalFlip(p=0.5),
    # Reduced Rotations: Less likely to flip the ground truth
    transforms.RandomRotation(degrees=15),
    
    # Milder Color Jitter: Less distortion on high-res textures
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05),
    
    # Removed RandomVerticalFlip, RandomGrayscale, and GaussianBlur to stabilize consistency loss
    
    transforms.ToTensor(),
    transforms.Normalize(mean_rgb, std_rgb),
])

# --- SOURCE (NWPU) TRANSFORMS ---

# 🟦 Source transforms (NWPU) - Kept mostly the same, standard source training
src_train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean_rgb, std_rgb),
])
src_test_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean_rgb, std_rgb),
])

# 📂 Load datasets from folders
src_train = datasets.ImageFolder(root="./data/source_nwpu_split/train", transform=src_train_tf) # <--- CHECK/ADJUST THIS PATH
src_test  = datasets.ImageFolder(root="./data/source_nwpu_split/test",  transform=src_test_tf) # <--- CHECK/ADJUST THIS PATH

# Using the correct path for target splits:
tgt_train_raw = datasets.ImageFolder(root="./data/target_nasc_split/train", transform=None) # <--- CHECK/ADJUST THIS PATH
tgt_test  = datasets.ImageFolder(root="./data/target_nasc_split/test",  transform=tgt_weak_tf) # <--- CHECK/ADJUST THIS PATH

print(f"Source (NWPU) train={len(src_train)}, test={len(src_test)}")
print(f"Target (NaSC)     train={len(tgt_train_raw)}, test={len(tgt_test)}")

# 🔁 Dual-view wrapper for target training (Unchanged)
class TwoViewRemoteSensing(Dataset):
    def __init__(self, base_ds, weak_tf, strong_tf):
        self.base = base_ds
        self.weak_tf = weak_tf
        self.strong_tf = strong_tf
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        img, _ = self.base[idx]
        xw = self.weak_tf(img)
        xs = self.strong_tf(img)
        return xw, xs, -1  # dummy label

tgt_train = TwoViewRemoteSensing(tgt_train_raw, tgt_weak_tf, tgt_strong_tf)

# 🚚 DataLoaders (Requires CFG class to be defined)
# This part is correct assuming CFG is defined:
src_train_loader = DataLoader(src_train, batch_size=CFG.src_batch_size, shuffle=True,  num_workers=CFG.num_workers, pin_memory=True)
src_test_loader  = DataLoader(src_test,  batch_size=CFG.src_batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)
tgt_unl_loader   = DataLoader(tgt_train, batch_size=CFG.sf_batch_size, shuffle=True,  num_workers=0, pin_memory=True)
tgt_test_loader  = DataLoader(tgt_test,  batch_size=CFG.sf_batch_size, shuffle=False, num_workers=0, pin_memory=True)

# Note: The DataLoader section is commented out as it relies on the CFG class, which is outside this block.
# Assuming CFG is defined as per the previous steps, the DataLoader definitions are logically correct.

In [9]:
#@title 🧠 Model for RGB images (3 channels)
import torch
import torch.nn as nn
from torchvision import models

class ResNetFeatureExtractor(nn.Module):
    """
    ResNet-18 backbone with a custom feature projection head
    for the CAB-SFDA framework.
    """
    def __init__(self, num_classes=6, feat_dim=128, in_channels=3, use_pretrained=True):
        super().__init__()
        
        # Load pre-trained ResNet-18
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if use_pretrained else None)
        
        # 1. Feature Extractor (Backbone)
        # We use the layers up to the AdaptiveAvgPool2d
        # The default ResNet-18 final feature size is 512
        self.features = nn.Sequential(
            self.backbone.conv1,
            self.backbone.bn1,
            self.backbone.relu,
            self.backbone.maxpool,
            self.backbone.layer1,
            self.backbone.layer2,
            self.backbone.layer3,
            self.backbone.layer4,
            self.backbone.avgpool # Global Average Pooling
        )
        
        # Check if the input channels match (e.g., if using multi-spectral data)
        if in_channels != 3:
            # Replace the first convolutional layer to handle different channel counts
            self.backbone.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)

        # 2. Projection Head (Feature Head for SFDA)
        # ResNet-18 outputs 512 features after avgpool
        self.proj_dim = self.backbone.fc.in_features # This will be 512
        
        # Add a custom projection layer to reduce dimensionality to feat_dim (128)
        self.feature_head = nn.Sequential(
            nn.Linear(self.proj_dim, feat_dim),
            nn.BatchNorm1d(feat_dim),
            nn.ReLU(inplace=True)
        )
        
        # 3. Classifier Head (Prediction Head)
        self.classifier = nn.Linear(feat_dim, num_classes)
        
        # Initialize custom layers
        nn.init.kaiming_normal_(self.feature_head[0].weight, mode='fan_out', nonlinearity='relu')
        nn.init.constant_(self.feature_head[0].bias, 0)
        nn.init.kaiming_normal_(self.classifier.weight, mode='fan_out', nonlinearity='relu')
        nn.init.constant_(self.classifier.bias, 0)
        
    def forward(self, x, return_feat=False):
        # 1. Extract Backbone Features
        f_map = self.features(x)         # [B, 512, 1, 1]
        f_map = f_map.view(f_map.size(0), -1) # [B, 512]
        
        # 2. Project Features to SFDA dimension
        f = self.feature_head(f_map)     # [B, feat_dim=128]
        
        # 3. Classify
        logits = self.classifier(f)      # [B, num_classes]
        
        if return_feat:
            # Return final classification logits and the projected features 'f'
            return logits, f 
        return logits

def accuracy(model, loader):
    """Computes classification accuracy."""
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            # The model is now ResNetFeatureExtractor
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
    return 100.0 * correct / max(1, total)

# Initialize model
# NOTE: CFG and device must be defined in the notebook context (as they usually are)
src_model = ResNetFeatureExtractor(
    num_classes=CFG.num_classes, 
    feat_dim=CFG.feat_dim, 
    in_channels=3 # Assuming RGB
).to(device)

print(src_model)
print("✅ Model updated to pre-trained ResNet-18 for better domain adaptation.")

ResNetFeatureExtractor(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine

In [10]:
#@title 🏋️ Train source model on EuroSAT (supervised)
def train_source(model, train_loader, test_loader, epochs=CFG.src_epochs, lr=CFG.src_lr):
    # ⬇️ IMPROVEMENT: Using SGD with Momentum for better generalization 
    # and stable convergence, especially with ResNet backbones.
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    best = -1
    
    # ⚠️ Check if the model is the powerful ResNet and confirm the LR is appropriate (1e-3 is fine for ResNet fine-tuning)

    for ep in range(1, epochs + 1):
        model.train()
        pbar = tqdm(train_loader, desc=f"Source epoch {ep}/{epochs}")
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        acc_src = accuracy(model, test_loader)
        best = max(best, acc_src)
        print(f"[Source] Epoch {ep}: test acc = {acc_src:.2f}% (best {best:.2f}%)")

    return model

# ⬇️ IMPROVEMENT: Initialize the model using the powerful ResNetFeatureExtractor
src_model = ResNetFeatureExtractor(
    num_classes=CFG.num_classes, 
    feat_dim=CFG.feat_dim, 
    in_channels=3 # Assuming RGB
).to(device)

print(f"Model initialized: {type(src_model).__name__}")

# Train source model
# Note: Ensure src_train_loader and src_test_loader are available in your environment.
src_model = train_source(src_model, src_train_loader, src_test_loader, epochs=CFG.src_epochs, lr=CFG.src_lr)

# Save checkpoint
src_ckpt_path = os.path.join(CFG.out_dir, "source_model.pth")
torch.save(src_model.state_dict(), src_ckpt_path)
print("✅ Saved source model to:", src_ckpt_path)

# Evaluate on target BEFORE adaptation
# Note: Ensure tgt_test_loader is available in your environment.
src_only_acc_on_target = accuracy(src_model, tgt_test_loader)
print(f"📊 Source-only accuracy on target test: {src_only_acc_on_target:.2f}%")

Model initialized: ResNetFeatureExtractor


Source epoch 1/50: 100%|██████████| 77/77 [00:37<00:00,  2.08it/s, loss=0.0934]


[Source] Epoch 1: test acc = 94.12% (best 94.12%)


Source epoch 2/50: 100%|██████████| 77/77 [00:25<00:00,  2.99it/s, loss=0.0228]


[Source] Epoch 2: test acc = 95.92% (best 95.92%)


...

[Source] Epoch 48: test acc = 98.20% (best 98.29%)


Source epoch 49/50: 100%|██████████| 77/77 [00:25<00:00,  2.99it/s, loss=0.0002]


[Source] Epoch 49: test acc = 98.10% (best 98.29%)


Source epoch 50/50: 100%|██████████| 77/77 [00:25<00:00,  2.97it/s, loss=0.0016]


[Source] Epoch 50: test acc = 98.15% (best 98.29%)
✅ Saved source model to: ././cab_sfda_ckpts_nwpu_to_NaSC-TG2-RGB\source_model.pth
📊 Source-only accuracy on target test: 43.85%


In [ ]:
#@title 🔁 CAB-SFDA adaptation 

import torch
import torch.nn as nn
import torch.nn.functional as F
import os
from torch.optim.lr_scheduler import CosineAnnealingLR
# Note: tqdm, CFG, device, ResNetFeatureExtractor, accuracy must be available in the environment

# --- Loss Functions ---
def entropy(p, eps=1e-8):
    return -(p * (p + eps).log()).sum(dim=1)

def info_max_loss(logits):
    p = torch.softmax(logits, dim=1)
    ent_per = entropy(p)
    ent_mean = entropy(p.mean(dim=0, keepdim=True))
    return ent_per.mean() - ent_mean.mean()

def kl_divergence_with_temperature(logits_a, logits_b, T=CFG.temp_cons):
    pa = torch.log_softmax(logits_a / T, dim=1)
    return F.kl_div(pa, torch.softmax(logits_b / T, dim=1), reduction='batchmean')

# --- Class Memory (for Prototype-based losses) ---
class ClassMemory:
    def __init__(self, num_classes, feat_dim, momentum=CFG.proto_momentum, eps=1e-6):
        self.num_classes = num_classes
        self.feat_dim = feat_dim
        self.m = momentum
        self.eps = eps
        self.prototypes = torch.zeros(num_classes, feat_dim, device=device)
        self.counts = torch.zeros(num_classes, device=device)
        self.initialized = torch.zeros(num_classes, dtype=torch.bool, device=device)

    def update(self, feats, labels):
        for c in range(self.num_classes):
            idx = (labels == c).nonzero(as_tuple=False).flatten()
            if idx.numel() == 0: continue
            fmean = feats[idx].mean(dim=0)
            if not self.initialized[c]:
                self.prototypes[c] = fmean.detach()
                self.initialized[c] = True
            else:
                self.prototypes[c] = self.m * self.prototypes[c] + (1 - self.m) * fmean.detach()
            self.counts[c] += idx.numel()

    def get_weights(self):
        inv = 1.0 / torch.sqrt(self.counts + self.eps)
        inv = inv / inv.sum().clamp_min(self.eps) * self.num_classes
        return inv.detach()

    def proto_loss(self, feats, labels):
        if feats.size(0) == 0: return torch.tensor(0.0, device=feats.device)
        protos = self.prototypes[labels]
        return F.mse_loss(feats, protos)

# --- CRITICAL HELPER FUNCTION ---
def freeze_bn(m):
    """Sets BatchNorm layers to eval mode to preserve source statistics."""
    if isinstance(m, nn.BatchNorm2d):
        m.eval()

# --- MODEL INITIALIZATION AND LOADING ---

src_ckpt_path = os.path.join(CFG.out_dir, "source_model.pth")
print("📁 Loading source model from:", src_ckpt_path)

if not os.path.exists(src_ckpt_path):
    raise FileNotFoundError(f"Source model checkpoint not found at {src_ckpt_path}.")

# Initialize student model with ResNetFeatureExtractor
student = ResNetFeatureExtractor(
    num_classes=CFG.num_classes, 
    feat_dim=CFG.feat_dim, 
    in_channels=3 
).to(device)

student.load_state_dict(torch.load(src_ckpt_path, map_location=device))
print("✅ Loaded pre-trained source model weights")

# Freeze ResNet BatchNorm layers & Classifier
student.features.apply(freeze_bn) 
for p in student.classifier.parameters():
    p.requires_grad_(False)
print("🔒 Frozen ResNet BatchNorm layers & Classifier layer")


# --- Optimizer with Layer-Wise Learning Rates (SGD) ---
optim_params = [
    # ⬇️ FINAL STABILIZATION: Feature Factor 0.1
    {'params': student.features.parameters(), 'lr': CFG.sf_lr * 0.1}, 
    # Projection Head (Aggressive)
    {'params': student.feature_head.parameters(), 'lr': CFG.sf_lr * 1.0},
]
optim_params = [p for p in optim_params if any(param.requires_grad for param in p['params'])]

optimizer = torch.optim.SGD(optim_params, momentum=0.9, weight_decay=1e-4)

# Initialize Cosine LR Scheduler 
scheduler = CosineAnnealingLR(optimizer, T_max=CFG.sf_epochs, eta_min=1e-6) 
print(f"✅ Initialized Cosine LR Scheduler for {CFG.sf_epochs} epochs.")

# Initialize Class Memory
memory = ClassMemory(num_classes=CFG.num_classes, feat_dim=CFG.feat_dim)

# Warmup epochs for guidance activation
warmup_epochs = 5 
best_tgt = -1.0

print(f"🚀 Starting CAB-SFDA adaptation for {CFG.sf_epochs} epochs with {warmup_epochs} warmup epochs...")

# --- ADAPTATION LOOP ---
for ep in range(1, CFG.sf_epochs + 1):
    student.train()
    student.features.apply(freeze_bn) 
    pbar = tqdm(tgt_unl_loader, desc=f"CAB-SFDA epoch {ep}/{CFG.sf_epochs}")

    # --- Dynamic Confidence Threshold & Loss Weight ---
    rampup_ratio = 0.0
    if ep < warmup_epochs:
        current_conf_thresh = 0.0 # No filtering
        loss_weight_ramp_factor = 0.0 # No guidance loss contribution
    elif ep <= CFG.rampup_epochs: 
        # Linearly ramp up Conf_Thresh and Loss Weight from 0.0 to Max
        rampup_ratio = (ep - warmup_epochs) / (CFG.rampup_epochs - warmup_epochs + 1e-6)
        current_conf_thresh = CFG.conf_thresh * min(1.0, rampup_ratio)
        loss_weight_ramp_factor = min(1.0, rampup_ratio)
    else:
        current_conf_thresh = CFG.conf_thresh # Fixed maximum threshold
        loss_weight_ramp_factor = 1.0 # Fixed maximum loss weight
    
    # -----------------------------------

    total_loss_im = total_loss_cons = total_loss_pl = total_loss_proto = 0
    total_batches = 0

    for xw, xs, _ in pbar:
        xw, xs = xw.to(device), xs.to(device)

        # Forward pass
        logits_w, feats_w = student(xw, return_feat=True)
        logits_s, feats_s = student(xs, return_feat=True)

        # 1. InfoMax loss
        loss_im = info_max_loss(logits_w)

        # 2. Consistency loss (calculated for logging, but not used in loss)
        kl_ws = kl_divergence_with_temperature(logits_w, logits_s)
        kl_sw = kl_divergence_with_temperature(logits_s, logits_w)
        loss_cons = 0.5 * (kl_ws + kl_sw)

        # Pseudo-labeling
        p_w = torch.softmax(logits_w, dim=1)
        conf, y_hat = p_w.max(dim=1)
        # Use the dynamic threshold for filtering
        mask = conf.ge(current_conf_thresh) 

        # 3. Pseudo-label CE loss (after warm-up)
        if ep >= warmup_epochs and mask.sum() > 0:
            class_weights = memory.get_weights()
            per_sample_ce = F.cross_entropy(logits_s[mask], y_hat[mask], reduction="none")
            w = class_weights[y_hat[mask]]
            loss_pl = (per_sample_ce * w).mean()
        else:
            loss_pl = torch.tensor(0.0, device=device)

        # 4. Prototype attraction loss (after warm-up and full memory init)
        if ep >= warmup_epochs and mask.sum() > 0 and memory.initialized.all(): 
            loss_proto = memory.proto_loss(feats_w[mask], y_hat[mask])
        else:
            loss_proto = torch.tensor(0.0, device=device)

        # Total loss
        # Use the ramp factor to soften the activation of PL and Proto losses
        loss = (
            CFG.lambda_im * loss_im +
            0.0 * loss_cons +  # Hard-coded to 0.0 for stability
            loss_weight_ramp_factor * (CFG.lambda_pl * loss_pl + CFG.lambda_proto * loss_proto)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Update memory with confident samples
        if ep >= warmup_epochs and mask.sum() > 0:
            memory.update(feats_w.detach()[mask], y_hat.detach()[mask])

        # Accumulate for logging
        total_loss_im += loss_im.item(); total_loss_cons += loss_cons.item()
        total_loss_pl += loss_pl.item(); total_loss_proto += loss_proto.item()
        total_batches += 1

        pbar.set_postfix(
            IM=f"{loss_im.item():.3f}", Cons=f"{loss_cons.item():.3f}",
            PL=f"{loss_pl.item():.3f}", Proto=f"{loss_proto.item():.3f}",
            Masked=f"{mask.sum().item()}/{xw.size(0)}"
        )

    # Evaluate on target test set
    student.eval() # Set model to evaluation mode for accuracy calculation
    acc_tgt = accuracy(student, tgt_test_loader)
    
    # Conditional save for Early Stopping
    if acc_tgt > best_tgt:
        best_tgt = acc_tgt
        tgt_best_ckpt_path = os.path.join(CFG.out_dir, "student_adapted_BEST_CAB_SFDA.pth")
        torch.save(student.state_dict(), tgt_best_ckpt_path)
        print(f"🎉 Saved new best model checkpoint to: {tgt_best_ckpt_path} with acc {best_tgt:.2f}%")
        
    student.train() # Set back to training mode
    student.features.apply(freeze_bn) 

    # Step the scheduler at the end of the epoch
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    avg_im = total_loss_im / total_batches
    avg_cons = total_loss_cons / total_batches
    avg_pl = total_loss_pl / total_batches
    avg_proto = total_loss_proto / total_batches

    print(f"[CAB-SFDA] Epoch {ep}: Target test acc = {acc_tgt:.2f}% (best {best_tgt:.2f}%), LR: {current_lr:.6f}, Conf_Thresh: {current_conf_thresh:.2f}, Loss_Ramp: {loss_weight_ramp_factor:.2f}")
    print(f"           Losses - IM: {avg_im:.3f}, Cons: {avg_cons:.3f}, PL: {avg_pl:.3f}, Proto: {avg_proto:.3f}")

# Save adapted model
tgt_ckpt_path = os.path.join(CFG.out_dir, "student_adapted_CAB_SFDA.pth")
torch.save(student.state_dict(), tgt_ckpt_path)
print("✅ Saved adapted model to:", tgt_ckpt_path)


# Final evaluation
student.eval()
final_acc = accuracy(student, tgt_test_loader)
print(f"🎯 Final adapted model accuracy on target: {final_acc:.2f}%")

# Calculate improvement
try:
    print(f"📈 Improvement over source-only: {final_acc - src_only_acc_on_target:+.2f}%")
except NameError:
    print("📊 Source-only accuracy not available for comparison")

In [ ]:
#import torch, numpy as np
import matplotlib.pyplot as plt
import os
import torch.nn.functional as F # Ensure F is imported for evaluation (though not strictly needed in evaluate_with_cm)

# NOTE: The CFG class and the ResNetFeatureExtractor class must be defined in the preceding cells.

def evaluate_with_cm(model, loader, num_classes=CFG.num_classes):
    """Evaluates model performance and returns overall accuracy, per-class accuracy, and confusion matrix."""
    model.eval()
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device); y = y.to(device)
            logits = model(x)
            pred = logits.argmax(1)
            for t, p in zip(y.view(-1), pred.view(-1)):
                cm[int(t.item()), int(p.item())] += 1

    per_class_acc = []
    for c in range(num_classes):
        total_c = cm[c].sum()
        # Ensure total_c is float before division
        acc_c = (cm[c, c] / float(total_c)) if total_c > 0 else 0.0
        per_class_acc.append(acc_c)

    overall_acc = cm.trace() / max(1, cm.sum())
    return overall_acc, per_class_acc, cm

def try_load_model(path):
    """
    Initializes and loads the ResNetFeatureExtractor model from a checkpoint path.
    """
    if os.path.exists(path):
        # ⬇️ CRITICAL FIX: Use the correct model class (ResNetFeatureExtractor)
        m = ResNetFeatureExtractor(
            num_classes=CFG.num_classes, 
            feat_dim=CFG.feat_dim, 
            in_channels=3 # Assuming RGB
        ).to(device)
        
        m.load_state_dict(torch.load(path, map_location=device))
        
        # ⚠️ NOTE: Freezing BN should be done if you plan to continue training, but for evaluation, it's not strictly necessary.
        # However, to be safe, we will leave the model in its adapted state (BN frozen/eval mode).
        m.features.apply(lambda m: m.eval() if isinstance(m, nn.BatchNorm2d) else None)
        
        return m
    return None

# Evaluate models
results = {}

# 1) In-memory adapted model (The 'student' model after the SFDA loop)
try:
    _ = student # Check if student object exists
    acc, pc, cm = evaluate_with_cm(student, tgt_test_loader)
    results["Adapted (in-memory student)"] = (acc, pc, cm)
    print("✅ Evaluated in-memory adapted model.")
except NameError:
    print("⚠️ In-memory 'student' model not found. Trying to load from checkpoint.")
    pass

# 2) Saved adapted checkpoint
# ⬇️ CRITICAL FIX: Use the correct checkpoint name
adapt_ckpt = os.path.join(CFG.out_dir, "student_adapted_CAB_SFDA.pth")
m_adapt = try_load_model(adapt_ckpt)
if m_adapt is not None and "Adapted (in-memory student)" not in results:
    acc, pc, cm = evaluate_with_cm(m_adapt, tgt_test_loader)
    results["Adapted (ckpt)"] = (acc, pc, cm)
    print(f"✅ Evaluated adapted model from checkpoint: {adapt_ckpt}")

# 3) Source-only checkpoint
# ⬇️ CRITICAL FIX: Use the correct checkpoint name
source_ckpt = os.path.join(CFG.out_dir, "source_model.pth")
m_src = try_load_model(source_ckpt)
if m_src is not None:
    acc, pc, cm = evaluate_with_cm(m_src, tgt_test_loader)
    results["Source-only on NWPU"] = (acc, pc, cm)
    print(f"✅ Evaluated source-only model from checkpoint: {source_ckpt}")


# Print report
def pct(x): return f"{100.0*x:.2f}%"
# ⬇️ Ensure labels match the class maps used previously
shared_labels = ["Beach", "CircularFarmland", "Cloud", "Desert", "Forest", "Mountain", "Rectangular farmland", "Residential", "River", "Snowberg"] 

print("\n=== Target Performance Report (NWPU → NaSC-TG2-RGB ===")
for name, (acc, pc, cm) in results.items():
    print(f"\n[{name}]")
    print(f"Overall accuracy: {pct(acc)}")
    print("Per-class accuracy:")
    for i, a in enumerate(pc):
        print(f"   {shared_labels[i]:<12}: {pct(a)}")
    print("\nConfusion Matrix:")
    print("Rows = True labels | Columns = Predicted labels")
    # Improved CM printing for better alignment
    cm_header = "    " + " ".join(f"{lbl[:4]:>4}" for lbl in shared_labels)
    print(cm_header)
    for i, row in enumerate(cm):
        row_str = " ".join(f"{val:4d}" for val in row)
        print(f"{shared_labels[i][:4]:>4} {row_str}")

# Show confusion matrix for best model
if results:
    best_name = max(results.items(), key=lambda kv: kv[1][0])[0]
    _, _, best_cm = results[best_name]
    
    # ⬇️ Added normalization for better visualization
    best_cm_normalized = best_cm.astype('float') / best_cm.sum(axis=1)[:, np.newaxis]
    
    fig = plt.figure(figsize=(8, 7)) # Increased size for readability
    plt.imshow(best_cm_normalized, interpolation='nearest', cmap='viridis') # Use 'viridis' or 'Blues'
    plt.title(f"Confusion Matrix (Normalized) — {best_name}")
    plt.xlabel("Predicted Label"); plt.ylabel("True Label")
    
    tick_marks = np.arange(CFG.num_classes)
    plt.xticks(tick_marks, shared_labels, rotation=45, ha="right")
    plt.yticks(tick_marks, shared_labels)
    plt.colorbar(label='Recall (Proportion of Class Correctly Classified)')
    
    # Add text labels for counts
    for i in range(CFG.num_classes):
        for j in range(CFG.num_classes):
            plt.text(j, i, f"{best_cm[i, j]}", 
                     ha="center", va="center", 
                     color="white" if best_cm_normalized[i, j] > best_cm_normalized.max() / 2 else "black",
                     fontsize=9)
            
    plt.tight_layout()
    plt.show()
    
else:
    print("No models found to evaluate. Train/adapt first, then re-run this cell.")

In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import numpy as np
import os

def extract_features(model, loader):
    """Extract feature embeddings and labels from a model using return_feat=True."""
    model.eval()
    all_feats, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            _, feats = model(x, return_feat=True)
            all_feats.append(feats.cpu())
            all_labels.append(y.cpu())
    return torch.cat(all_feats), torch.cat(all_labels)

def plot_embeddings(feats, labels, label_names, title, method="tsne", save_path=None):
    """Reduce features to 2D and plot them with color-coded labels."""
    if method == "tsne":
        reducer = TSNE(n_components=2, perplexity=30, init='pca', random_state=42)
    elif method == "pca":
        reducer = PCA(n_components=2)
    else:
        raise ValueError("Method must be 'tsne' or 'pca'")

    emb = reducer.fit_transform(feats.numpy())
    plt.figure(figsize=(8, 5))
    for label in torch.unique(labels):
        idx = (labels == label)
        plt.scatter(emb[idx, 0], emb[idx, 1], label=label_names[label.item()], alpha=0.6, s=10)
    plt.title(title)
    plt.legend(loc='best', fontsize=8)
    plt.axis('off')
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"✅ Saved plot to: {save_path}")
    plt.show()


# --- Execution Block ---

# Label mappings (NWPU -> NaSC-TG2-RGB labels)
category_names = ["Beach", "CircularFarmland", "Cloud", "Desert", "Forest", "Mountain", "Rectangular farmland", "Residential", "River", "Snowberg"] 
domain_names = ["Source (NaSC-TG2-RGB)", "Target (NWPU)"] 

# Load trained model
model = ResNetFeatureExtractor(num_classes=CFG.num_classes).to(device)
model.load_state_dict(torch.load(src_ckpt_path, map_location=device))
model.eval()
'''
# 🌟 CRITICAL FIX: Dynamically determine feat_dim to resolve AttributeError
if not hasattr(CFG, 'feat_dim'):
    with torch.no_grad():
        # Use a dummy batch from the loader to determine the feature size
        x_dummy, _ = next(iter(tgt_test_loader))
        x_dummy = x_dummy.to(device)
        _, feats_dummy = model(x_dummy, return_feat=True)
        CFG.feat_dim = feats_dummy.shape[1]
        print(f"Feature Dimension (feat_dim) dynamically set to: {CFG.feat_dim}")

'''
# Extract features from source and target
src_feats, src_labels = extract_features(model, src_test_loader)
tgt_feats, tgt_labels = extract_features(model, tgt_test_loader)

# 1. Domain visualization (Checking Domain Shift)
src_domain_labels = torch.zeros_like(src_labels)
tgt_domain_labels = torch.ones_like(tgt_labels)
domain_feats = torch.cat([src_feats, tgt_feats])
domain_labels = torch.cat([src_domain_labels, tgt_domain_labels])
plot_embeddings(domain_feats, domain_labels, domain_names,
                title="Feature Distribution by Domain (Source-Only Model)",
                method="tsne",
                save_path=os.path.join(CFG.out_dir, "tsne_domain_before.png"))

# 2. Category visualization (Checking Class Separability)
category_feats = torch.cat([src_feats, tgt_feats])
category_labels = torch.cat([src_labels, tgt_labels])
plot_embeddings(category_feats, category_labels, category_names,
                title="Feature Distribution by Category (Source-Only Model)",
                method="tsne",
                save_path=os.path.join(CFG.out_dir, "tsne_category_before.png"))
# Extract features
adapt_model = ResNetFeatureExtractor(num_classes=CFG.num_classes).to(device)
src_feats, src_labels = extract_features(src_model, src_test_loader)
tgt_feats_pre, tgt_labels_pre = extract_features(src_model, tgt_test_loader)
tgt_feats_post, tgt_labels_post = extract_features(adapt_model, tgt_test_loader)

# Domain labels
src_domain_labels = torch.zeros_like(src_labels)
tgt_domain_labels_pre = torch.ones_like(tgt_labels_pre)
tgt_domain_labels_post = torch.ones_like(tgt_labels_post)
# Plot domain alignment
plot_embeddings(torch.cat([src_feats, tgt_feats_pre]),
                torch.cat([src_domain_labels, tgt_domain_labels_pre]),
                domain_names,
                title="Domain Alignment (Before Adaptation)",
                method="tsne",
                save_path=os.path.join(CFG.out_dir, "tsne_domain_before.png"))

plot_embeddings(torch.cat([src_feats, tgt_feats_post]),
                torch.cat([src_domain_labels, tgt_domain_labels_post]),
                domain_names,
                title="Domain Alignment (After Adaptation)",
                method="tsne",
                save_path=os.path.join(CFG.out_dir, "tsne_domain_after.png"))

# Plot category alignment
plot_embeddings(torch.cat([src_feats, tgt_feats_pre]),
                torch.cat([src_labels, tgt_labels_pre]),
                category_names,
                title="Category Alignment (Before Adaptation)",
                method="tsne",
                save_path=os.path.join(CFG.out_dir, "tsne_category_before.png"))

plot_embeddings(torch.cat([src_feats, tgt_feats_post]),
                torch.cat([src_labels, tgt_labels_post]),
                category_names,
                title="Category Alignment (After Adaptation)",
                method="tsne",
                save_path=os.path.join(CFG.out_dir, "tsne_category_after.png"))